In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.00 0.95      0.08
SUVpeak                   0.29 0.59      0.76
TLG                       0.22 0.64      0.65
age                       1.60 0.21      2.28
cavum_oris                0.00 1.00      0.01
charlson                  0.07 0.79      0.34
female                    0.17 0.68      0.56
histgrade_high            1.26 0.26      1.93
hpv_related               4.31 0.04      4.72
hypopharynx               0.00 0.99      0.01
larynx                    0.00 0.98      0.02
oropharynx                0.00 0.99      0.02
pack_years                0.71 0.40      1.32
uicc8_III-IV              0.45 0.50      0.99


In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.02 0.88      0.18
SUVpeak                   0.56 0.46      1.13
TLG                       0.06 0.81      0.30
age                       1.58 0.21      2.26
cavum_oris                0.08 0.77      0.37
charlson                  0.03 0.86      0.22
female                    0.35 0.56      0.85
histgrade_high            0.95 0.33      1.60
hpv_related               3.02 0.08      3.60
hypopharynx               0.10 0.75      0.41
larynx                    1.46 0.23      2.14
oropharynx                0.96 0.33      1.61
pack_years                0.39 0.53      0.92
uicc8_III-IV              0.08 0.78      0.36


In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = RobustScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [25]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [26]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [27]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [28]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.497635,1,0,1,0,0,1,0.0,0,-0.747766,0.0,0.600320,0.075008,0.231096
1,-0.473435,0,0,0,0,1,0,0.0,1,0.159774,0.0,-0.690705,-0.466877,-0.387041
2,-0.113739,0,1,0,0,0,1,0.0,1,0.610629,1.0,-0.308082,0.641923,0.210342
3,0.826312,0,0,0,0,1,0,0.0,1,0.494088,0.0,-0.987020,-0.399206,-0.398539
4,0.596634,0,0,0,0,1,0,0.0,1,1.007387,0.0,-0.641777,-0.293211,-0.315926
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.000000,0,0,1,0,0,1,1.0,0,-0.747766,0.0,-0.109388,-0.294765,-0.236855
135,0.671213,0,0,1,0,0,1,1.0,0,-0.747766,1.0,-0.399297,1.027319,0.352293
136,-0.235838,0,0,1,0,0,1,1.0,1,0.560275,0.0,0.518153,-0.059989,0.075327
137,0.421516,0,0,1,0,0,1,1.0,1,1.620942,1.0,-0.353490,0.464201,0.118722


In [29]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [30]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.436476,0,0,1,0,0,1,1,1,-0.747766,0,0.732497,1.361702,1.615732
1,-0.436476,0,0,1,0,0,0,0,0,-0.085444,1,-0.172482,-0.121272,-0.153327
2,-0.436476,0,0,1,0,0,0,0,1,-0.549070,1,0.463784,0.062665,0.140609
3,0.045320,1,0,0,0,1,1,0,1,0.742458,1,-0.199405,0.072763,-0.076742
4,0.768012,0,0,1,0,0,1,1,1,1.206084,0,-0.041772,0.705364,0.398213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.446816,1,0,0,0,1,0,0,0,1.073619,1,2.909606,-0.082431,0.685886
95,0.205918,0,0,0,0,1,0,0,1,5.014436,1,0.404287,0.010099,0.098289
96,0.205918,0,0,1,0,0,1,1,1,-0.747766,1,-0.156713,0.812913,0.358846
97,-0.516775,0,0,1,0,0,1,1,0,-0.747766,0,0.560744,0.251694,0.363804


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [31]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:01:31,223] A new study created in memory with name: no-name-4103015e-6f02-4a8e-a73a-f3af14f64981


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-13 17:01:31,375] A new study created in memory with name: no-name-3c8de404-5593-488e-8e3b-75c581a6123b


Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:01:31,372] Trial 0 finished with value: 0.6371137109369622 and parameters: {}. Best is trial 0 with value: 0.6371137109369622.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6371137109369622], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 31, 267340), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 31, 372661), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6371137109369622


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.25671409350000224
Fold 2 IBS: 0.6067543088664037
Fold 3 IBS: 0.168829941254425
Fold 4 IBS: 0.295979507187782
Fold 5 IBS: 0.21043512284090146
[I 2024-04-13 17:01:31,495] Trial 0 finished with value: 0.30774259472990284 and parameters: {}. Best is trial 0 with value: 0.30774259472990284.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.30774259472990284], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 31, 386859), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 31, 495863), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.30774259472990284


In [32]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [33]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.308


#### Test

In [34]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [35]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.563
IBS score: 0.291


In [36]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [37]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:01:31,572] A new study created in memory with name: no-name-357fc2b2-f859-47d5-8d67-2f0b750fe619


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.5348837209302325


[I 2024-04-13 17:01:31,647] A new study created in memory with name: no-name-0d07a9a0-b6ca-4391-87e4-12d4b19f3d10


Fold 3 C-index: 0.6085106382978723
Fold 4 C-index: 0.4695817490494297
Fold 5 C-index: 0.6223175965665236
[I 2024-04-13 17:01:31,645] Trial 0 finished with value: 0.5657838405704052 and parameters: {}. Best is trial 0 with value: 0.5657838405704052.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5657838405704052], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 31, 587592), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 31, 645205), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5657838405704052


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709360937505
Fold 2 IBS: 0.23203988427112568
Fold 3 IBS: 0.228981863568446
Fold 4 IBS: 0.24197478945971077
Fold 5 IBS: 0.2293955857787768
[I 2024-04-13 17:01:31,729] Trial 0 finished with value: 0.23592784333748687 and parameters: {}. Best is trial 0 with value: 0.23592784333748687.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784333748687], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 31, 663132), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 31, 729136), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784333748687


In [39]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.566
train_ibs:  0.236


#### Test

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.584


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:01:31,853] A new study created in memory with name: no-name-a639e3e2-43c9-4ef6-a67e-26c1016899e4


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5387596899224806
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6768060836501901


[I 2024-04-13 17:01:32,013] A new study created in memory with name: no-name-84ff22df-cee7-4181-8b91-0872ada49cf5


Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:01:32,011] Trial 0 finished with value: 0.6347812481157444 and parameters: {}. Best is trial 0 with value: 0.6347812481157444.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6347812481157444], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 31, 873570), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 32, 11507), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6347812481157444


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.261310757127916
Fold 2 IBS: 0.2631821342546185
Fold 3 IBS: 0.17043677379023145
Fold 4 IBS: 0.29558039313374845
Fold 5 IBS: 0.2102637591042142
[I 2024-04-13 17:01:32,184] Trial 0 finished with value: 0.24015476348214576 and parameters: {}. Best is trial 0 with value: 0.24015476348214576.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.24015476348214576], datetime_start=datetime.datetime(2024, 4, 13, 17, 1, 32, 28436), datetime_complete=datetime.datetime(2024, 4, 13, 17, 1, 32, 184122), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.24015476348214576


In [45]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.635
train_ibs:  0.24


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:01:32,378] A new study created in memory with name: no-name-442307e1-b18c-4acf-a31c-777856b27c42


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.630901287553648
[I 2024-04-13 17:01:32,524] Trial 0 finished with value: 0.635556441914194 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.635556441914194.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:32,660] Trial 1 finished with value: 0.6388304491343872 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6388304491343872.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:32,798] Trial 2 finished with value: 0.6388304491343872 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:35,454] Trial 24 finished with value: 0.6380336363853832 and parameters: {'l1_ratio': 0.16645977610301255}. Best is trial 1 with value: 0.6388304491343872.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:01:35,590] Trial 25 finished with value: 0.6379720800356747 and parameters: {'l1_ratio': 0.44908780975162055}. Best is trial 1 with value: 0.6388304491343872.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:35,727] Trial 26 finished with value: 0.6388304491343872 and parameters: {'l1_ratio': 0.208020460948690

Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:01:38,556] Trial 49 finished with value: 0.6372116237619104 and parameters: {'l1_ratio': 0.46569877065454696}. Best is trial 1 with value: 0.6388304491343872.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:38,684] Trial 50 finished with value: 0.6388304491343872 and parameters: {'l1_ratio': 0.374354691474329}. Best is trial 1 with value: 0.6388304491343872.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:01:38,820] Trial 51 finished with value: 0.6388304491343872 and parameters: {'l1_ratio': 0.24691530243536003

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.26102620358767353
Fold 2 IBS: 0.26312657631566294
Fold 3 IBS: 0.1703241267537586
Fold 4 IBS: 0.29545196784653605
Fold 5 IBS: 0.21005729223038072
[I 2024-04-13 17:01:44,919] Trial 0 finished with value: 0.2399972333468024 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2399972333468024.
Fold 1 IBS: 0.2588837003522285
Fold 2 IBS: 0.26303985644748085
Fold 3 IBS: 0.17022526620181144
Fold 4 IBS: 0.29526478881241885
Fold 5 IBS: 0.20973282979278876
[I 2024-04-13 17:01:45,057] Trial 1 finished with value: 0.23942928832134566 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.23942928832134566.
Fold 1 IBS: 0.2587173379871686
Fold 2 IBS: 0.2629671855344548
Fold 3 IBS: 0.1701580742235107
Fold 4 IBS: 0.29522557084537643
Fold 5 IBS: 0.20968044356486212
[I 2024-04-13 17:01:45,198] Trial 2 finished with value: 0.2393497224310745 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2393497224310745.

Fold 1 IBS: 0.2578444032559643
Fold 2 IBS: 0.2320308875725676
Fold 3 IBS: 0.2263064516043076
Fold 4 IBS: 0.2543205914847007
Fold 5 IBS: 0.2253616485058363
[I 2024-04-13 17:01:47,581] Trial 25 finished with value: 0.23917279648467532 and parameters: {'l1_ratio': 0.07330522822697523}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.25866786192150326
Fold 2 IBS: 0.26317276643326787
Fold 3 IBS: 0.17012862694920441
Fold 4 IBS: 0.29521598225183937
Fold 5 IBS: 0.20965163859069955
[I 2024-04-13 17:01:47,729] Trial 26 finished with value: 0.2393673752293029 and parameters: {'l1_ratio': 0.20309774849254153}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.25772258764105377
Fold 2 IBS: 0.23200587852032056
Fold 3 IBS: 0.22681478866412372
Fold 4 IBS: 0.25189660165357836
Fold 5 IBS: 0.22608260794741947
[I 2024-04-13 17:01:47,806] Trial 27 finished with value: 0.23890449288529916 and parameters: {'l1_ratio': 0.056847137315710346}. Best is trial 13 with value: 0.235924213

Fold 2 IBS: 0.2630950559724324
Fold 3 IBS: 0.17012112790234593
Fold 4 IBS: 0.2951804917569898
Fold 5 IBS: 0.20960653389035813
[I 2024-04-13 17:01:50,145] Trial 50 finished with value: 0.23931353090052068 and parameters: {'l1_ratio': 0.1780409676995512}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.24684804227027152
Fold 2 IBS: 0.23202540235002309
Fold 3 IBS: 0.22871579490331465
Fold 4 IBS: 0.2430652229173587
Fold 5 IBS: 0.22896941427316414
[I 2024-04-13 17:01:50,204] Trial 51 finished with value: 0.2359247753428264 and parameters: {'l1_ratio': 0.006012844952470507}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.2579790079397853
Fold 2 IBS: 0.23206724896637737
Fold 3 IBS: 0.225842762182722
Fold 4 IBS: 0.2564681131417791
Fold 5 IBS: 0.20951696835839173
[I 2024-04-13 17:01:50,298] Trial 52 finished with value: 0.23637482011781108 and parameters: {'l1_ratio': 0.08964599767605695}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.247173963377

Fold 4 IBS: 0.295120472001801
Fold 5 IBS: 0.20961116135161323
[I 2024-04-13 17:01:52,270] Trial 75 finished with value: 0.24396048890908711 and parameters: {'l1_ratio': 0.15221237807038437}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.2450599313994901
Fold 2 IBS: 0.23199413137287353
Fold 3 IBS: 0.2274473459644907
Fold 4 IBS: 0.2488507488904063
Fold 5 IBS: 0.22701038289912787
[I 2024-04-13 17:01:52,326] Trial 76 finished with value: 0.2360725081052777 and parameters: {'l1_ratio': 0.038228682177325266}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.2589012977495234
Fold 2 IBS: 0.262979026769936
Fold 3 IBS: 0.1702531063542334
Fold 4 IBS: 0.29524255554919454
Fold 5 IBS: 0.20971018848551673
[I 2024-04-13 17:01:52,469] Trial 77 finished with value: 0.23941723498168083 and parameters: {'l1_ratio': 0.2786795986705474}. Best is trial 13 with value: 0.2359242139249683.
Fold 1 IBS: 0.2581887635298997
Fold 2 IBS: 0.23212913543895755
Fold 3 IBS: 0.225292059844445

In [51]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.639
train_ibs:  0.236


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.28621072101688444)

test_cindex : 0.564


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.004533947374774427)

test_ibs:  0.229


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:01:54,338] A new study created in memory with name: no-name-dafbc98e-fec3-45b9-991e-681ff16002eb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6201550387596899
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.6673003802281369
Fold 5 C-index: 0.6072961373390557
[I 2024-04-13 17:01:55,552] Trial 0 finished with value: 0.6187977301006736 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6187977301006736.
Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6472868217054264
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.6094420600858369
[I 2024-04-13 17:01:56,384] Trial 1 finished with value: 0.6237488136710192 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 2 C-index: 0.6937984496124031
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6759656652360515
[I 2024-04-13 17:02:05,323] Trial 17 finished with value: 0.6895409260222489 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 83, 'oob_score': True, 'max_samples': 0.9817072304699028, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2138706147057923, 'warm_start': True}. Best is trial 17 with value: 0.6895409260222489.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.6824034334763949
[I 2024-04-13 17:02:05,533] Trial 18 finished with value: 0.6896987117465925 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 15, 'min_samples_leaf': 17, 'max_depth': 4, 'n_estimators': 105, 'oob_score': True, 'max_samples': 0.9862374149206906, 'max_features': 'log2', 'min_weigh

Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.7510729613733905
[I 2024-04-13 17:02:11,247] Trial 32 finished with value: 0.71347233498625 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 14, 'n_estimators': 482, 'oob_score': True, 'max_samples': 0.7213418356032232, 'max_features': None, 'min_weight_fraction_leaf': 0.005268431893676387, 'warm_start': True}. Best is trial 32 with value: 0.71347233498625.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.7553648068669528
[I 2024-04-13 17:02:12,151] Trial 33 finished with value: 0.7143454416096479 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 15, 'n_estimators': 497, 'oob_score': True, 'max_samples': 0.7317060450615778, 

...

Fold 1 IBS: 0.2370347091364531
Fold 2 IBS: 0.21865616933268361
Fold 3 IBS: 0.2136904590169095
Fold 4 IBS: 0.2330616724900941
Fold 5 IBS: 0.21522492222501335
[I 2024-04-13 17:03:11,783] Trial 32 finished with value: 0.22353358644023075 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 310, 'oob_score': True, 'max_samples': 0.8040571874379057, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.18786523282060735}. Best is trial 32 with value: 0.22353358644023075.
Fold 1 IBS: 0.2369596498960518
Fold 2 IBS: 0.21919381104819285
Fold 3 IBS: 0.21367435784125732
Fold 4 IBS: 0.2331924481484345
Fold 5 IBS: 0.21510595787220624
[I 2024-04-13 17:03:13,301] Trial 33 finished with value: 0.22362524496122854 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 309, 'oob_score': True, 'max_samples': 0.7910445247744765, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.21452585999296125
[I 2024-04-13 17:03:40,060] Trial 47 finished with value: 0.22331758578489697 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 332, 'oob_score': True, 'max_samples': 0.8648447148686852, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19908924320632904}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23571677073756844
Fold 2 IBS: 0.21862106487993588
Fold 3 IBS: 0.21456356649548577
Fold 4 IBS: 0.23300957311771886
Fold 5 IBS: 0.2147097776323396
[I 2024-04-13 17:03:41,959] Trial 48 finished with value: 0.2233241505726097 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 9, 'n_estimators': 339, 'oob_score': True, 'max_samples': 0.85048001681846, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.21469064698337334}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23559067856645097
Fold 2 IBS: 0.218738

Fold 1 IBS: 0.24251844789708726
Fold 2 IBS: 0.2210411248095999
Fold 3 IBS: 0.2174884582534811
Fold 4 IBS: 0.23327796476633547
Fold 5 IBS: 0.21325865662478627
[I 2024-04-13 17:04:09,770] Trial 63 finished with value: 0.225516930470258 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 440, 'oob_score': True, 'max_samples': 0.8434390256503708, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.13262761517233745}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.24154578994370174
Fold 2 IBS: 0.22210454661611326
Fold 3 IBS: 0.21614753754341642
Fold 4 IBS: 0.23303912664616094
Fold 5 IBS: 0.21337979688122055
[I 2024-04-13 17:04:11,926] Trial 64 finished with value: 0.2252433595261226 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 421, 'oob_score': True, 'max_samples': 0.9071936553420485, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.21423498837277127
[I 2024-04-13 17:04:35,975] Trial 78 finished with value: 0.22373591837101642 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 10, 'n_estimators': 346, 'oob_score': True, 'max_samples': 0.8854068847594949, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.23765891161800223}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23570867974207424
Fold 2 IBS: 0.22060193373952594
Fold 3 IBS: 0.2144464193518562
Fold 4 IBS: 0.23476296957407278
Fold 5 IBS: 0.21578644913062256
[I 2024-04-13 17:04:37,409] Trial 79 finished with value: 0.22426129030763034 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 291, 'oob_score': True, 'max_samples': 0.9991782770608287, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2796131771286568}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.25860616056552466
Fold 2 IBS: 0.247

Fold 1 IBS: 0.2364707434109151
Fold 2 IBS: 0.21910149125141531
Fold 3 IBS: 0.21420284080337593
Fold 4 IBS: 0.23438012728527638
Fold 5 IBS: 0.21479897289302796
[I 2024-04-13 17:05:13,287] Trial 94 finished with value: 0.22379083512880213 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 430, 'oob_score': True, 'max_samples': 0.892312681224388, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2167154554420358}. Best is trial 84 with value: 0.22302339534037285.
Fold 1 IBS: 0.23950861114017571
Fold 2 IBS: 0.22006606579763324
Fold 3 IBS: 0.2159169717512573
Fold 4 IBS: 0.2336883258166932
Fold 5 IBS: 0.21371153072549798
[I 2024-04-13 17:05:16,034] Trial 95 finished with value: 0.22457830104625148 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 481, 'oob_score': True, 'max_samples': 0.8276360496389354, 'max_features': 'log2', 'min_weight_fraction_leaf'

In [57]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.76
train_ibs:  0.223


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [60]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=5, max_leaf_nodes=8,
                     max_samples=0.9313711932374781, min_samples_split=12,
                     min_weight_fraction_leaf=0.011633867937315842,
                     n_estimators=214, random_state=123, warm_start=True)

test_cindex:  0.633


RandomSurvivalForest(max_depth=6, max_features='log2', max_leaf_nodes=20,
                     max_samples=0.8096737689830545, min_samples_leaf=2,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.19628098798162147,
                     n_estimators=453, oob_score=True, random_state=123)

test_ibs:  0.216


In [61]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [63]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:05:26,915] A new study created in memory with name: no-name-8cf49f5a-eb8b-49d8-9bc2-05e8ff684224


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7489361702127659
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.6695278969957081
[I 2024-04-13 17:05:27,447] Trial 0 finished with value: 0.6810141068632278 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6810141068632278.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:05:28,367] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.6627906976744186
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 17:05:39,865] Trial 16 finished with value: 0.6718042314769685 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.6867027610976928.
Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.6453488372093024
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.6566523605150214
[I 2024-04-13 17:05:40,342] Trial 17 finished with value: 0.6672980040406395 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7682403433476395
[I 2024-04-13 17:05:47,405] Trial 31 finished with value: 0.7161536582914563 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9402340730195752, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 28 with value: 0.7175262582650991.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.721030042918455
[I 2024-04-13 17:05:47,678] Trial 32 finished with value: 0.6961121726190391 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features':

...

Fold 1 IBS: 0.24654710532719232
Fold 2 IBS: 0.23226075421028072
Fold 3 IBS: 0.22941657391206893
Fold 4 IBS: 0.24156072638580853
Fold 5 IBS: 0.23017807225897458
[I 2024-04-13 17:07:15,366] Trial 30 finished with value: 0.235992646418865 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 12, 'max_depth': 15, 'n_estimators': 225, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.46244718326068934, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23751141455085345
Fold 2 IBS: 0.2151581315756588
Fold 3 IBS: 0.20565940925991258
Fold 4 IBS: 0.22542469872025206
Fold 5 IBS: 0.21030193870732267
[I 2024-04-13 17:07:17,868] Trial 31 finished with value: 0.2188111185627999 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 500, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5

Fold 1 IBS: 0.23579057506686668
Fold 2 IBS: 0.21781214726067985
Fold 3 IBS: 0.2066254011517932
Fold 4 IBS: 0.22753811203645144
Fold 5 IBS: 0.2131782901601682
[I 2024-04-13 17:07:50,034] Trial 45 finished with value: 0.22018890513519188 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 367, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.7526038702156265, 'min_weight_fraction_leaf': 0.19723066794982302}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23864304620933718
Fold 2 IBS: 0.2216376067444548
Fold 3 IBS: 0.21798786862601383
Fold 4 IBS: 0.2317000936375164
Fold 5 IBS: 0.21855619191514636
[I 2024-04-13 17:07:51,970] Trial 46 finished with value: 0.2257049614264937 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 306, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8402191

Fold 3 IBS: 0.2102452041922602
Fold 4 IBS: 0.22890896945323644
Fold 5 IBS: 0.21433456345556318
[I 2024-04-13 17:08:12,181] Trial 60 finished with value: 0.22226672714026327 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 16, 'n_estimators': 74, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9528336631358474, 'min_weight_fraction_leaf': 0.23484468684942653}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23541239037810044
Fold 2 IBS: 0.2157600681640186
Fold 3 IBS: 0.20665073702737224
Fold 4 IBS: 0.22705259747647938
Fold 5 IBS: 0.2114858383168534
[I 2024-04-13 17:08:14,345] Trial 61 finished with value: 0.21927232627256482 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 397, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6192920945584268, 'min_weight_fraction_leaf': 0.1326170753826787

Fold 1 IBS: 0.23749669116970254
Fold 2 IBS: 0.21417891364052918
Fold 3 IBS: 0.2066472918191204
Fold 4 IBS: 0.22577332010612122
Fold 5 IBS: 0.20791194308464916
[I 2024-04-13 17:08:45,796] Trial 75 finished with value: 0.2184016319640245 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 346, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6413926717278569, 'min_weight_fraction_leaf': 0.0007388507314896667}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23958121408927258
Fold 2 IBS: 0.21329051656876732
Fold 3 IBS: 0.20685348469946407
Fold 4 IBS: 0.2243628744610485
Fold 5 IBS: 0.2076885994354418
[I 2024-04-13 17:08:48,766] Trial 76 finished with value: 0.21835533785079883 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 356, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6

Fold 1 IBS: 0.23843719064657817
Fold 2 IBS: 0.22064066243477068
Fold 3 IBS: 0.216117076423818
Fold 4 IBS: 0.23057174337958733
Fold 5 IBS: 0.21778600564999445
[I 2024-04-13 17:09:31,172] Trial 90 finished with value: 0.22471053570694974 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 274, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.42143690848778903, 'min_weight_fraction_leaf': 0.05305541913345835}. Best is trial 83 with value: 0.21749549175321342.
Fold 1 IBS: 0.23636584853089593
Fold 2 IBS: 0.2160409129934387
Fold 3 IBS: 0.20518796152371307
Fold 4 IBS: 0.22686151762465062
Fold 5 IBS: 0.21138354158709388
[I 2024-04-13 17:09:33,895] Trial 91 finished with value: 0.21916795645195847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 17, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 307, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.378

In [64]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [65]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.217


#### Test

In [66]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [67]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=16,
                   max_samples=0.8020091028727125, min_samples_leaf=1,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.002435943155287043,
                   n_estimators=277, random_state=123, warm_start=True)

C-index score: 0.584


ExtraSurvivalTrees(max_depth=6, max_features='auto', max_leaf_nodes=16,
                   max_samples=0.44903497421134486, min_samples_leaf=4,
                   min_samples_split=9,
                   min_weight_fraction_leaf=0.018023565445760347,
                   n_estimators=351, oob_score=True, random_state=123)

IBS: 0.223


In [68]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [69]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:09:55,716] A new study created in memory with name: no-name-f4050f5c-e3ca-450d-8642-325b3a47b141


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:10:12,067] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:10:19,519] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:15:45,271] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6059172111077464.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:16:30,906] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:27:26,468] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.6190116704947725.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:29:07,219] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 19:28:52,987] Trial 44 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9479336661285427, 'learning_rate': 0.0149361535238727, 'dropout_rate': 0.25335630680617827, 'n_estimators': 238, 'criterion': 'squared_error', 'ccp_alpha': 0.5490150526266808, 'min_weight_fraction_leaf': 0.4126953497238681, 'max_features': 'auto', 'min_impurity_decrease': 0.00078866474186123, 'validation_fraction': 0.9438667820963776, 'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 3}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24678393645753394
Fold 2 IBS: 0.23045362112354736
Fold 3 IBS: 0.22798640772703402
Fold 4 IBS: 0.24133177588235574
Fold 5 IBS: 0.22819468131537443
[I 2024-04-13 19:29:12,684] Trial 45 finished with value: 0.2349500845011691 and parameters: {'subsample': 0.5852424762732177, 'learning_rate': 0.065638506610498

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 19:31:32,346] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 19:32:11,383] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 19:35:48,099] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7279386202897999, 'learning_rate': 0.09984676062907398, 'dropout_rate': 0.12777796849617917, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 1, 'min_impurity_decrease': 0.00014974488025406402, 'validation_fraction': 0.6772907402354433, 'min_samples_split': 8, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 3}. Best is trial 65 with value: 0.2330518973544506.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 19:35:51,943] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8919877355885673, 'learning_rate': 0.08526801927485488, 'dropout_rate': 0.1968183597819077, 'n_estimators': 45, 'cri

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.2419747714592711
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 19:41:33,895] Trial 78 finished with value: 0.2359278435123307 and parameters: {'subsample': 0.9982285661999424, 'learning_rate': 0.09191504218778693, 'dropout_rate': 0.15698726385072695, 'n_estimators': 197, 'criterion': 'squared_error', 'ccp_alpha': 0.24807516981478814, 'min_weight_fraction_leaf': 0.20156232220196313, 'max_features': 0.1, 'min_impurity_decrease': 1.657879625668623e-05, 'validation_fraction': 0.743279291715464, 'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 11}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24690614034739003
Fold 2 IBS: 0.23156150181893284
Fold 3 IBS: 0.22844484931909595
Fold 4 IBS: 0.24167312234765576
Fold 5 IBS: 0.22876368497640898
[I 2024-04-13 19:41:39,209] Trial 79 finished with value: 0.23546985976189672 and parameters: {'s

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:11:56,579] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9483409389571185, 'learning_rate': 0.08684995291290751, 'dropout_rate': 0.261199231115212, 'n_estimators': 33, 'criterion': 'squared_error', 'ccp_alpha': 0.262026471967215, 'min_weight_fraction_leaf': 0.30628737845191195, 'max_features': 0.1, 'min_impurity_decrease': 6.315494653924259e-06, 'validation_fraction': 0.7881463591488752, 'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 15, 'max_depth': 4}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-13 20:12:08,538] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9683729112775776, 'le

In [70]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [71]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.642
train_ibs:  0.232


#### Test

In [72]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [73]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0019473494055538935,
                                 criterion='squared_error',
                                 dropout_rate=0.43613571687055647,
                                 learning_rate=0.09542136158814303,
                                 max_depth=20, max_features='sqrt',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=5.255459802548745e-07,
                                 min_samples_leaf=15, min_samples_split=10,
                                 min_weight_fraction_leaf=0.36743918461462793,
                                 n_estimators=66, random_state=123,
                                 subsample=0.569175138136116,
                                 validation_fraction=0.7087056989518657)

C-index score: 0.669


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008911991304216094,
                                 criterion='squared_error',
                                 dropout_rate=0.1333613034046263,
                                 learning_rate=0.09420039979456486,
                                 max_depth=14, max_features=0.1,
                                 max_leaf_nodes=19,
                                 min_impurity_decrease=2.4912576060277585e-06,
                                 min_samples_leaf=17, min_samples_split=7,
                                 min_weight_fraction_leaf=0.2444025476799629,
                                 n_estimators=141, random_state=123,
                                 subsample=0.9998829056403309,
                                 validation_fraction=0.6909543967814417)

IBS: 0.224


In [74]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [76]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 20:13:16,888] A new study created in memory with name: no-name-35a91697-7bed-48bc-b7a7-05c017d1b19a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.501937984496124
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6201716738197425
[I 2024-04-13 20:13:19,991] Trial 0 finished with value: 0.559767342351139 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.559767342351139.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 20:13:42,875] Trial 1 finished with value: 0.5594249857365082 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.559767342351139.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.501937984496124
Fold 3 C-index: 0.5595744680851064
Fold 4 C-i

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5058139534883721
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.5893536121673004
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 20:16:42,354] Trial 19 finished with value: 0.5811290338880302 and parameters: {'subsample': 0.34314044045637715, 'dropout_rate': 0.6899537536342808, 'n_estimators': 106, 'learning_rate': 0.08294848495969458}. Best is trial 14 with value: 0.6064776215330693.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5193798449612403
Fold 3 C-index: 0.5872340425531914
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:16:59,615] Trial 20 finished with value: 0.5930736662113076 and parameters: {'subsample': 0.2628688080573812, 'dropout_rate': 0.19350037184585095, 'n_estimators': 415, 'learning_rate': 0.09794678439106413}. Best is trial 14 with value: 0.6064776215330693.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.5872340425531914


Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:20:04,850] Trial 37 finished with value: 0.5889632056991776 and parameters: {'subsample': 0.25024731314917675, 'dropout_rate': 0.16068871497947684, 'n_estimators': 478, 'learning_rate': 0.011979304832032835}. Best is trial 14 with value: 0.6064776215330693.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5232558139534884
Fold 3 C-index: 0.6127659574468085
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 20:20:10,659] Trial 38 finished with value: 0.5980968738897682 and parameters: {'subsample': 0.10048762833294078, 'dropout_rate': 0.2950640677363966, 'n_estimators': 190, 'learning_rate': 0.048252223497364057}. Best is trial 14 with value: 0.6064776215330693.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.5914893617021276
Fold 4 C-index: 0.5893536121673004
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:20:14,949] Trial 39 finished with value: 0.59630828

...

Fold 3 IBS: 0.22552825499814835
Fold 4 IBS: 0.2665867816858836
Fold 5 IBS: 0.209166894601268
[I 2024-04-13 20:40:46,554] Trial 19 finished with value: 0.23929136261847264 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.23291148620212007.
Fold 1 IBS: 0.2446002185537661
Fold 2 IBS: 0.23463757481110972
Fold 3 IBS: 0.2201655501901051
Fold 4 IBS: 0.2439567647593348
Fold 5 IBS: 0.22296335334244682
[I 2024-04-13 20:40:48,070] Trial 20 finished with value: 0.23326469233135252 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.23291148620212007.
Fold 1 IBS: 0.24513654608372853
Fold 2 IBS: 0.23383949946417815
Fold 3 IBS: 0.22211599951175628
Fold 4 IBS: 0.2430985991111111
Fold 5 IBS: 0.22475950483168106
[I 2024-04-13 20:40:49,481] Trial 21 finis

Fold 3 IBS: 0.2345343216426952
Fold 4 IBS: 0.27480342364574306
Fold 5 IBS: 0.20820368705343698
[I 2024-04-13 20:41:42,271] Trial 38 finished with value: 0.2447954102876631 and parameters: {'subsample': 0.703355276038254, 'dropout_rate': 0.13272164755980653, 'n_estimators': 113, 'learning_rate': 0.03521923736878125}. Best is trial 17 with value: 0.23291148620212007.
Fold 1 IBS: 0.25062091351496935
Fold 2 IBS: 0.25679440758337285
Fold 3 IBS: 0.23094671738894873
Fold 4 IBS: 0.2723938763086342
Fold 5 IBS: 0.21265849486129715
[I 2024-04-13 20:41:49,144] Trial 39 finished with value: 0.24468288193144444 and parameters: {'subsample': 0.628838778564896, 'dropout_rate': 0.48841341315505027, 'n_estimators': 243, 'learning_rate': 0.013931445007643331}. Best is trial 17 with value: 0.23291148620212007.
Fold 1 IBS: 0.25217715275845426
Fold 2 IBS: 0.2677872216513955
Fold 3 IBS: 0.23151320239683454
Fold 4 IBS: 0.2638382103949051
Fold 5 IBS: 0.21394023230043377
[I 2024-04-13 20:41:52,274] Trial 40 fin

Fold 3 IBS: 0.31066778988567284
Fold 4 IBS: 0.2953220518192507
Fold 5 IBS: 0.33637252588000854
[I 2024-04-13 20:43:17,427] Trial 57 finished with value: 0.33829999103224595 and parameters: {'subsample': 0.9973261031802491, 'dropout_rate': 0.565616782693081, 'n_estimators': 204, 'learning_rate': 0.06795033768066108}. Best is trial 51 with value: 0.23225942239148312.
Fold 1 IBS: 0.24856912303953135
Fold 2 IBS: 0.2648811882881005
Fold 3 IBS: 0.22584711134343113
Fold 4 IBS: 0.23864085166272536
Fold 5 IBS: 0.21522732252870758
[I 2024-04-13 20:43:24,039] Trial 58 finished with value: 0.23863311937249918 and parameters: {'subsample': 0.9493350320664393, 'dropout_rate': 0.5038560979284157, 'n_estimators': 243, 'learning_rate': 0.011811490867438759}. Best is trial 51 with value: 0.23225942239148312.
Fold 1 IBS: 0.247706117411449
Fold 2 IBS: 0.2620466566474926
Fold 3 IBS: 0.22424657230600534
Fold 4 IBS: 0.23607552600936632
Fold 5 IBS: 0.21552378922282525
[I 2024-04-13 20:43:33,528] Trial 59 fini

Fold 2 IBS: 0.23862969480175472
Fold 3 IBS: 0.21707217888229136
Fold 4 IBS: 0.2689140303226001
Fold 5 IBS: 0.20748843095039043
[I 2024-04-13 20:44:59,131] Trial 76 finished with value: 0.23503834206771917 and parameters: {'subsample': 0.21986786119949991, 'dropout_rate': 0.2640930463660313, 'n_estimators': 99, 'learning_rate': 0.03129844324263236}. Best is trial 73 with value: 0.23169935333488173.
Fold 1 IBS: 0.2402848847898735
Fold 2 IBS: 0.23438829712812412
Fold 3 IBS: 0.21121418177496804
Fold 4 IBS: 0.25933659550694627
Fold 5 IBS: 0.20938395486478042
[I 2024-04-13 20:45:02,215] Trial 77 finished with value: 0.23092158281293845 and parameters: {'subsample': 0.10027825939376077, 'dropout_rate': 0.3127569715361363, 'n_estimators': 132, 'learning_rate': 0.020102710084664722}. Best is trial 77 with value: 0.23092158281293845.
Fold 1 IBS: 0.2410805558867413
Fold 2 IBS: 0.23530499933287927
Fold 3 IBS: 0.21320884353140648
Fold 4 IBS: 0.2608386827506965
Fold 5 IBS: 0.2080759472906538
[I 2024

Fold 1 IBS: 0.2427492136521167
Fold 2 IBS: 0.23183948041552568
Fold 3 IBS: 0.21311208627262304
Fold 4 IBS: 0.25051259661582037
Fold 5 IBS: 0.21204976836178402
[I 2024-04-13 20:45:37,318] Trial 95 finished with value: 0.23005262906357396 and parameters: {'subsample': 0.15264452640327397, 'dropout_rate': 0.17030680435941772, 'n_estimators': 41, 'learning_rate': 0.04617763984593193}. Best is trial 91 with value: 0.22918776724187162.
Fold 1 IBS: 0.24543496525606437
Fold 2 IBS: 0.23140021504927716
Fold 3 IBS: 0.22560829397671991
Fold 4 IBS: 0.24490841093042917
Fold 5 IBS: 0.2256979875863117
[I 2024-04-13 20:45:38,086] Trial 96 finished with value: 0.23460997455976046 and parameters: {'subsample': 0.24358576777347152, 'dropout_rate': 0.14401520560687697, 'n_estimators': 8, 'learning_rate': 0.04481151765552657}. Best is trial 91 with value: 0.22918776724187162.
Fold 1 IBS: 0.24270476272257596
Fold 2 IBS: 0.23369089769487017
Fold 3 IBS: 0.21321195455178632
Fold 4 IBS: 0.24908496368563687
Fold 

In [77]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [78]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.607
train_ibs:  0.229


#### Test

In [79]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [80]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.11467940028544912,
                                              learning_rate=0.09056873407930205,
                                              n_estimators=436,
                                              random_state=123,
                                              subsample=0.10185920727711952)

C-index score: 0.575


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.2315449509874271,
                                              learning_rate=0.046086009528404755,
                                              n_estimators=48, random_state=123,
                                              subsample=0.12997376659838994)

IBS: 0.239


In [81]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [82]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.791,1.0
Randomsurvivalforest,0.760,2.0
GradientBoosting,0.642,3.0
CoxElastic,0.639,4.0
CoxPH,0.637,5.0
CoxLasso,0.635,6.0
ComponentwiseGradientBoosting,0.607,7.0
CoxRidge,0.566,8.0


In [83]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.217,1.0
Randomsurvivalforest,0.223,2.0
ComponentwiseGradientBoosting,0.229,3.0
GradientBoosting,0.232,4.0
CoxRidge,0.236,5.5
CoxElastic,0.236,5.5
CoxLasso,0.240,7.0
CoxPH,0.308,8.0


In [84]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.669,1.0
Randomsurvivalforest,0.633,2.0
CoxRidge,0.584,3.5
ExtraSurvivalTrees,0.584,3.5
ComponentwiseGradientBoosting,0.575,5.0
CoxLasso,0.566,6.0
CoxElastic,0.564,7.0
CoxPH,0.563,8.0


In [85]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.216,1.0
ExtraSurvivalTrees,0.223,2.0
GradientBoosting,0.224,3.0
CoxRidge,0.229,4.5
CoxElastic,0.229,4.5
ComponentwiseGradientBoosting,0.239,6.0
CoxLasso,0.288,7.0
CoxPH,0.291,8.0


In [86]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/robust/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_robust_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [87]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-13
